# MiniProject02
# HealthGuard Insurance

## Setup and data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Download from Kaggle
try:
    import kagglehub
    path = kagglehub.dataset_download("mirichoi0218/insurance")
    df = pd.read_csv(f"{path}/insurance.csv")
except Exception:
    df = pd.read_csv("insurance.csv")

df.head()

## Load and profile the data

We used `head`, `shape`, `info`, `describe`, and `value_counts` to assess size, structure, quality.


In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
# Categorical columns
for col in ["sex", "smoker", "region"]:
    print(col, "->", df[col].value_counts().to_dict())

## Data cleaning

In [ ]:
# Missing values and duplicates
print("Missing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Drop the single duplicate row and keep raw df untouched
df_clean = df.drop_duplicates().reset_index(drop=True)
print("Rows before:", len(df), "| after dropping duplicates:", len(df_clean))

# One-hot encode categoricals
df_enc = pd.get_dummies(df_clean, columns=["sex", "smoker", "region"], drop_first=True)
df_enc.head()

Dropping the duplicate row was fine since it didn't shrink the dataset in any meaningful way. We one-hot encoded some categories instead of label encoding them so no false ordering occurs. There was no missing value imputation for us to do because insa sum returned a value of 0.

## Requirement 1 - Understanding the customers

In [ ]:
# Plot 1 - Distribution of the charges
plt.figure(figsize=(7,4))
plt.hist(df_clean["charges"], bins=40, edgecolor="black")
plt.title("Distribution of medical charges")
plt.xlabel("charges"); plt.ylabel("count"); plt.show()

In [ ]:
# Plot 2 - Smoking is the biggest driver of charges
groups = [df_clean[df_clean["smoker"] == s]["charges"] for s in ["no", "yes"]]
plt.figure(figsize=(7,4))
plt.boxplot(groups, labels=["no", "yes"])
plt.title("Charges by smoking status")
plt.xlabel("smoker"); plt.ylabel("charges"); plt.show()

In [ ]:
# Plot 3 - Age vs charges colored by smoker status
plt.figure(figsize=(7,5))
for s, color in [("no", "tab:blue"), ("yes", "tab:red")]:
    sub = df_clean[df_clean["smoker"] == s]
    plt.scatter(sub["age"], sub["charges"], alpha=0.6, label=f"smoker={s}", color=color)
plt.title("Charges vs age (by smoking status)")
plt.xlabel("age"); plt.ylabel("charges"); plt.legend(); plt.show()

In [ ]:
# Plot 4 - BMI vs charges colored by smoker status. High BMI with smoking has the highest charges
plt.figure(figsize=(7,5))
for s, color in [("no", "tab:blue"), ("yes", "tab:red")]:
    sub = df_clean[df_clean["smoker"] == s]
    plt.scatter(sub["bmi"], sub["charges"], alpha=0.6, label=f"smoker={s}", color=color)
plt.title("Charges vs BMI (by smoking status)")
plt.xlabel("bmi"); plt.ylabel("charges"); plt.legend(); plt.show()

In [ ]:
# Plot 5 - Average charges by region
region_means = df_clean.groupby("region")["charges"].mean()
plt.figure(figsize=(7,4))
plt.bar(region_means.index, region_means.values)
plt.title("Average charges by region")
plt.xlabel("region"); plt.ylabel("mean charges"); plt.show()

Most insurance charges fall between 5,000 and 15,000. Individuals who smoke generally incur higher charges than non-smokers. Among people with similar ages and BMIs, smokers tend to have significantly higher charges than those who do not smoke. Additionally, the Southeast has the highest average charges while the Southwest has the lowest.

## Requirement 2 - Predict the medical charges

In [ ]:
# Shared split
X = df_enc.drop(columns=["charges"])
y = df_enc["charges"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaled versions for SVR, Ridge and Lasso
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train) 
X_test_s  = scaler.transform(X_test)

# Helper to score a model and append to the results table
results = []
def record(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    results.append({"Model": name, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2})
    print(f"{name:22s}  MAE={mae:9.1f}  RMSE={rmse:9.1f}  R2={r2:.3f}")

In [ ]:
# Model 1 - Simple Linear Regression - single best predictor was if someone was a smoker or not
corr = df_enc.corr(numeric_only=True)
best_feat = corr["charges"].drop("charges").abs().idxmax()
m1 = LinearRegression().fit(X_train[[best_feat]], y_train)
record(f"1. Simple Linear ({best_feat})", y_test, m1.predict(X_test[[best_feat]]))

In [ ]:
# Model 2 - Multiple Linear Regression
m2 = LinearRegression().fit(X_train, y_train)
record("2. Multiple Linear", y_test, m2.predict(X_test))

In [ ]:
# Model 3 - Polynomial Regression - R2 drops as degree grows which is a sign of overfitting
for d in [2, 3, 4]:
    poly = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(X_train, y_train)
    record(f"3. Polynomial d={d}", y_test, poly.predict(X_test))

In [ ]:
# Model 4  - Ridge Regression (L2) - alpha is tuned by trying a few values and keeping the best
alphas = [0.01, 0.1, 1, 10, 100]
best_alpha, best_r2, best_ridge = None, -np.inf, None
for a in alphas:
    ridge = Ridge(alpha=a).fit(X_train_s, y_train)
    r2 = r2_score(y_test, ridge.predict(X_test_s))
    print(f"alpha={a:<6} R2={r2:.4f}")
    if r2 > best_r2:
        best_alpha, best_r2, best_ridge = a, r2, ridge
print("Best Ridge alpha:", best_alpha)
record("4. Ridge (L2)", y_test, best_ridge.predict(X_test_s))

In [ ]:
# Model 5 - Lasso Regression (L1) - alpha is tuned by trying a few values and keeping the best
alphas = [0.1, 1, 10, 100]
best_alpha, best_r2, best_lasso = None, -np.inf, None
for a in alphas:
    lasso = Lasso(alpha=a, max_iter=10000).fit(X_train_s, y_train)
    r2 = r2_score(y_test, lasso.predict(X_test_s))
    print(f"alpha={a:<6} R2={r2:.4f}")
    if r2 > best_r2:
        best_alpha, best_r2, best_lasso = a, r2, lasso
print("Best Lasso alpha:", best_alpha)
record("5. Lasso (L1)", y_test, best_lasso.predict(X_test_s))

eliminated = [f for f, c in zip(X.columns, best_lasso.coef_) if abs(c) < 1e-6]
print("Features eliminated at best alpha:", eliminated)

# The best-scoring alpha above is small so Lasso keeps every feature
# To see Lasso's feature-selection behavior look at a stronger alpha
strong_lasso = Lasso(alpha=100, max_iter=10000).fit(X_train_s, y_train)
strong_elim = [f for f, c in zip(X.columns, strong_lasso.coef_) if abs(c) < 1e-6]
print("For comparison, at alpha=100 Lasso eliminates:", strong_elim)

In [ ]:
# Model 6 - Support Vector Regression
for k in ["linear", "rbf"]:
    svr = SVR(kernel=k, C=1000).fit(X_train_s, y_train)
    record(f"6. SVR ({k})", y_test, svr.predict(X_test_s))

In [ ]:
# Model 7 - Decision Tree Regression
tree_reg = DecisionTreeRegressor(max_depth=4, random_state=42).fit(X_train, y_train)
record("7. Decision Tree d=4", y_test, tree_reg.predict(X_test))

# The tree
plt.figure(figsize=(30, 12))
plot_tree(tree_reg, feature_names=list(X.columns), filled=True, rounded=True,
          fontsize=8, impurity=False, precision=0)
plt.title("Decision Tree Regressor (max_depth=4)")
plt.tight_layout()
plt.show()

In [ ]:
# Summary table - all models and sorted by R2
results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results_df

We recommend the decision tree model because it has the highest accuracy and is easy to interpret. The model breaks the data down into a series of simple decisions based on factors such as smoking status, BMI, age, and number of children. This makes it straightforward to follow how the model arrives at its predictions and understand which variables have the greatest impact on insurance charges.

## Requirement 3 - Flag the expensive customers

In [ ]:
# The binary target from the median threshold
threshold = df_clean["charges"].median()
print("Median charge (threshold):", round(threshold, 2))

# 1 is expensive and 0 is not
y_cls = (df_enc["charges"] > threshold).astype(int)   
X_cls = df_enc.drop(columns=["charges"])
print(y_cls.value_counts().to_dict())

In [ ]:
# Split
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.2, stratify=y_cls, random_state=42)

sc_cls = StandardScaler()
Xc_train_s = sc_cls.fit_transform(Xc_train)
Xc_test_s  = sc_cls.transform(Xc_test)

In [ ]:
# Training the three classifiers
models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000), True),
    "Decision Tree":        (DecisionTreeClassifier(max_depth=4, random_state=42), False),
    "Random Forest":        (RandomForestClassifier(n_estimators=100, random_state=42), False),
}

for name, (model, scaled) in models.items():
    if scaled:
        model.fit(Xc_train_s, yc_train); pred = model.predict(Xc_test_s)
    else:
        model.fit(Xc_train, yc_train);   pred = model.predict(Xc_test)

    print("="*55); print(name)
    print(classification_report(yc_test, pred, target_names=["Not expensive (0)", "Expensive (1)"]))

    cm = confusion_matrix(yc_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=["Not exp.", "Expensive"]).plot(cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix — {name}"); plt.show()